# Bank Churn Prediction — Simple Exploration

Notebook first: inspect → split → preprocess → train → evaluate → save.

**Dataset:** `demo_churn.csv` is synthetic and for teaching only.

## 1. Import libraries

In [1]:
from pathlib import Path
import json
import joblib # model serlzation
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score, recall_score, f1_score

## 2. Read the CSV

In [2]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "data" / "demo_churn.csv")
print("Rows and columns:", df.shape)
df.head()

Rows and columns: (1800, 9)


,CreditScore,Geography,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
0,378,Germany,23,7,79090.96,2,1,1,1
1,584,Germany,66,3,46206.47,2,1,1,1
2,577,France,59,2,12127.71,3,1,1,0
3,479,Germany,45,1,0.00,2,1,1,0
4,414,Germany,45,0,71994.00,1,0,0,1


## 3. Understand and clean the data

In [9]:
df.isna().sum()

CreditScore       0
Geography         0
Age               0
Tenure            0
Balance           0
NumOfProducts     0
HasCrCard         0
IsActiveMember    0
Exited            0
dtype: int64

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
df['Exited'].value_counts()

Exited
0    1286
1     514
Name: count, dtype: int64

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CreditScore     1800 non-null   int64  
 1   Geography       1800 non-null   str    
 2   Age             1800 non-null   int64  
 3   Tenure          1800 non-null   int64  
 4   Balance         1800 non-null   float64
 5   NumOfProducts   1800 non-null   int64  
 6   HasCrCard       1800 non-null   int64  
 7   IsActiveMember  1800 non-null   int64  
 8   Exited          1800 non-null   int64  
dtypes: float64(1), int64(7), str(1)
memory usage: 137.2 KB


In [15]:
df.describe()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
count,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000
mean,599.182778,49.181667,4.934444,62930.115628,1.675556,0.700000,0.510556,0.285556
std,144.713055,18.224491,3.121966,60205.805400,0.701988,0.458385,0.500027,0.451804
min,350.000000,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,472.000000,33.000000,2.000000,0.000000,1.000000,0.000000,0.000000,0.000000
50%,589.000000,49.000000,5.000000,55452.480000,2.000000,1.000000,1.000000,0.000000
75%,728.000000,65.000000,8.000000,106121.407500,2.000000,1.000000,1.000000,1.000000
max,850.000000,80.000000,10.000000,275958.060000,4.000000,1.000000,1.000000,1.000000


## 4. Select 8 features and split data

In [4]:
FEATURES = [
    "CreditScore", "Geography", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember",
]
df = df.drop_duplicates().dropna(subset=FEATURES + ["Exited"])
X = df[FEATURES]
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training rows:", len(X_train), "Test rows:", len(X_test))

Training rows: 1440 Test rows: 360


In [ ]:
0 --> 70% ,1 --> 30%


xtrain --> y_train.  -->(80%) --> 30% 1 -- 50% 0
xtest --> ytest --> (20%). 20 0%


70% 30%  --> 80% training   --> 30% 1 50 %
70% 30%. --> 20% testing.   --> no data for class 1 , 20% data 

## 5. Preprocess Geography and train Random Forest

The Pipeline keeps `OneHotEncoder` and the model together, so the API will use the same preprocessing.

# model 
piepline --> geographical --? france germany spain --> component (encoding)(piepline) 

In [17]:
preprocessing = ColumnTransformer(
    [("country", OneHotEncoder(handle_unknown="ignore"), ["Geography"])],
    remainder="passthrough",
)
preprocessing

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('country', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e

In [18]:

model = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", RandomForestClassifier(n_estimators=50, random_state=42)),
])

In [19]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('country', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If 

In [20]:

model = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", RandomForestClassifier(n_estimators=50, random_state=42)),
])
model.fit(X_train, y_train)
print("Training complete!")

Training complete!


## 6. Evaluate the model

In [21]:
predictions = model.predict(X_test)
metrics = {
    "accuracy": round(accuracy_score(y_test, predictions), 3),
}
print(metrics)

{'accuracy': 0.728}


## 7. Save model + metrics

In [22]:
artifacts = ROOT / "artifacts"
artifacts.mkdir(exist_ok=True)

joblib.dump(model, artifacts / "churn_model.joblib")
with open(artifacts / "metrics.json", "w", encoding="utf-8") as file:
    json.dump({"model": "Random Forest", "metrics": metrics}, file, indent=2)

print("Model saved!")

Model saved!


## 8. Try one prediction

In [8]:
customer = X_test.iloc[[0]]
probability = model.predict_proba(customer)[0][1]
print("Churn probability:", round(float(probability), 3))

Churn probability: 0.66


## Next steps

Try the simple Python modules next: `app/data.py` → `app/preprocessing.py` → `app/training.py` → `app/evaluation.py`.

Run `uv run python train_model.py` and then build the API (`app/main.py`) and the UI (`ui/streamlit_app.py`).